In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [8]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()
len(docs)

9

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
splitted_docs = splitter.split_documents(docs)
len(splitted_docs)

25

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(
  documents=splitted_docs, embeddings=embeddings)

In [ ]:
same_records = vector_store.similarity_search("What is the patient's blood pressure?", k=2)

In [ ]:
@tool
def retriever_tool(query:str):
  """
this tool can help you to retrieve the relevant documents from the vector store based on the query. It will return the top 4 most similar documents to the query.
 
  """
  docs =vector_store.similarity_search(query, k=4)
  print(len(docs),docs[0].page_content)
  context = ""
  for doc in docs:
    context = doc.page_content + "\n\n"
  return context

In [ ]:
llm = ChatOpenAI(model="gpt-4o")

In [ ]:
system_prompt = """
You are a helpful assistant that can answer questions based on the context provided.
If the context does not contain the answer, respond with "I don't know."
"""

In [ ]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_message=system_prompt,
    agent_type="zero-shot-react-description",
)

In [ ]:
query = 'What is name of the patient, and what is the name of the doctors who treated the patient?'
response = agent.invoke({"message":[{"role":"user","content":query}]})

In [ ]:
result = response['message'][0]['content']
print(result)